# 04. 사진 바꾸기 — 배경 지우기 · 화질 4배 · 깊이

> 01~03 은 사진을 넣으면 **숫자(상자 · 확률)** 가 나왔다. 여기 세 모델은 사진을 넣으면 **사진(같은 크기의 숫자 판)** 이 나온다.
> 그래서 후처리가 "숫자 판을 다시 사진으로 만드는 일" 이 된다.

### 이 시간에 배우는 것
1. **정규화** — 평균을 빼고 표준편차로 나누는 이유
2. 출력 판(mask · depth)을 0~1 로 펴서 사진으로 바꾸기, **알파 합성**
3. 입력이 **두 개**인 모델 (화질 4배)
4. 상대 깊이(relative depth)의 뜻과 한계
5. 결과를 엮어 응용 — 배경 흐리기, 인물 사진 모드

| 모델 | 입력 | 전처리 | 출력 |
|---|---|---|---|
| U2Net (ONNX) | 320 x 320 RGB | ÷255 → (x − 평균) ÷ 표준편차 | 가림막(mask) 1채널 — 앞(물체)일수록 큰 값 |
| SR-1032 | 작은 사진 480 x 270 + 크게 늘린 사진 1920 x 1080 | 없음 (0~255) | 1920 x 1080, 값 0~1 |
| Depth Anything V2 Small | 518 x 518 RGB | U2Net 과 같은 정규화 | 상대 깊이 (클수록 가깝다) |

## 1. 준비 — 정규화 값
많은 사진 모델은 **ImageNet** 이라는 사진 묶음으로 미리 학습됐다. 그 사진들의 색별 평균 · 표준편차로 입력을 맞춰 줘야 배운 것과 같은 범위가 된다.

$$x' = \frac{x / 255 - \text{평균}}{\text{표준편차}} \qquad \text{평균} = (0.485,\ 0.456,\ 0.406),\ \text{표준편차} = (0.229,\ 0.224,\ 0.225) \ (\text{R, G, B})$$

결과는 대략 −2 ~ +2 사이가 된다.

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

def pick(*prefer):
    """먼저 적은 장치부터, 이 PC 에 있는 것을 고른다."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

DEVICE = pick("GPU", "CPU")                        # 큰 사진을 다루는 모델 — GPU
G = MODELS / "gan"
MEAN = np.array([0.485, 0.456, 0.406], np.float32)  # ImageNet 평균 · 표준편차 (RGB)
STD = np.array([0.229, 0.224, 0.225], np.float32)

def normalized(bgr, size):
    rgb = cv2.cvtColor(cv2.resize(bgr, (size, size)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return ((rgb - MEAN) / STD).transpose(2, 0, 1)[None].astype(np.float32)
print("장치:", DEVICE)

## 2. 사진

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. 배경 지우기 — U2Net
U2Net 은 사진에서 **눈에 띄는 것(salient object)** 을 찾아 "앞일수록 1, 뒤일수록 0" 인 판을 낸다. ONNX 파일이지만 OpenVINO 가 바로 연다.

**후처리**
1. 출력을 사진 안에서 0~1 로 편다 (min-max 정규화): $m' = \frac{m - \min}{\max - \min}$
2. 원래 사진 크기로 늘린다.
3. **알파 합성**: $\text{결과} = \text{사진} \times m + \text{배경} \times (1 - m)$ — m 이 1 인 곳은 사진, 0 인 곳은 배경.

In [ ]:
u2 = core.compile_model(core.read_model(G / "u2net.onnx"), DEVICE)
print("출력 수:", len(u2.outputs), "— 첫 출력이 가장 고운 가림막")

h, w = frame.shape[:2]
out = u2(normalized(frame, 320))[u2.output(0)]
mask = out.squeeze().astype(np.float32)
mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)   # ① 0~1 로 펴기
mask = cv2.resize(mask, (w, h))[..., None]                      # ② 원래 크기로, 채널 축 하나 추가 (곱하기 위해)
cut = (frame * mask + 255 * (1 - mask)).astype(np.uint8)        # ③ 배경을 흰색으로
show(np.hstack([frame, cv2.cvtColor((mask * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR), cut]))

In [ ]:
# 응용: 배경만 흐리게 — 배경 자리에 '흐린 원본' 을 합성한다
blur = cv2.GaussianBlur(frame, (0, 0), 12)
soft = (frame * mask + blur * (1 - mask)).astype(np.uint8)
hard = np.where(mask > 0.5, frame, blur).astype(np.uint8)      # 0.5 기준으로 딱 자르면 경계가 각진다
print("왼쪽: 부드러운 합성 | 오른쪽: 0.5 로 딱 자름")
show(np.hstack([soft, hard]))

## 4. 화질 4배 — 입력이 **두 개**인 모델
SR-1032 는 작은 사진(480 x 270)과, 그것을 그냥 크게 늘린 사진(1920 x 1080)을 **함께** 받는다.
그냥 늘린 사진에 "더해야 할 세부" 만 모델이 만든다 (잔차 학습). 입력 크기가 고정이라 비율이 다른 사진은 빈 곳을 채워 넣고 나중에 잘라 낸다.

입력이 여럿이면 `{이름: 배열}` 사전으로 넣는다.

In [ ]:
sr = core.compile_model(G / "single-image-super-resolution-1032.xml", DEVICE)
print("입력:", [(i.any_name, tuple(i.shape)) for i in sr.inputs])

def super_res(bgr):
    h, w = bgr.shape[:2]
    s = min(480 / w, 270 / h)
    nw, nh = max(1, int(w * s)), max(1, int(h * s))
    small = cv2.copyMakeBorder(cv2.resize(bgr, (nw, nh)), 0, 270 - nh, 0, 480 - nw, cv2.BORDER_REPLICATE)
    big = cv2.resize(small, (1920, 1080), interpolation=cv2.INTER_CUBIC)
    feed = {sr.inputs[0].any_name: small.transpose(2, 0, 1)[None].astype(np.float32),
            sr.inputs[1].any_name: big.transpose(2, 0, 1)[None].astype(np.float32)}
    out = sr(feed)[sr.output(0)].squeeze().transpose(1, 2, 0)   # CHW → HWC
    out = (out * 255).clip(0, 255).astype(np.uint8)             # 0~1 → 0~255
    return out[: nh * 4, : nw * 4], small[:nh, :nw]             # 채워 넣은 곳을 잘라 낸다

t0 = time.perf_counter()
big4, small = super_res(frame)
print(f"{small.shape[1]}x{small.shape[0]} → {big4.shape[1]}x{big4.shape[0]}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")
plain = cv2.resize(small, (big4.shape[1], big4.shape[0]), interpolation=cv2.INTER_CUBIC)
cy, cx = big4.shape[0] // 2, big4.shape[1] // 2          # 가운데 부분을 크게 비교
crop = lambda im: im[cy - 120: cy + 120, cx - 160: cx + 160]
print("왼쪽: 그냥 늘림(bicubic) | 오른쪽: 모델")
show(np.hstack([crop(plain), crop(big4)]))

## 5. 깊이 — 한 장의 사진에서 멀고 가까움
Depth Anything V2 는 사진 한 장으로 각 픽셀이 얼마나 가까운지 추정한다.
- **상대 깊이**: "이것이 저것보다 가깝다" 는 알지만 **몇 미터인지는 모른다**. 사진마다 기준이 달라서, 그 사진 안에서만 0~1 로 편다.
- 입력이 **518** 인 이유: 내부의 ViT 가 사진을 14 x 14 조각으로 나눈다. 518 = 14 × 37.

In [ ]:
depth = core.compile_model(G / "depth-v2s.xml", DEVICE)
d = depth(normalized(frame, 518))[depth.output(0)].squeeze().astype(np.float32)
d = cv2.resize(d, (w, h))
near = (d - d.min()) / (d.max() - d.min() + 1e-8)             # 이 사진 안에서만 0~1 (클수록 가깝다)
color = cv2.applyColorMap((near * 255).astype(np.uint8), cv2.COLORMAP_INFERNO)   # 밝을수록 가깝다
show(np.hstack([frame, color]))

In [ ]:
# 응용: 인물 사진 모드 — 가장 가까운 30% 만 선명하게
th = np.quantile(near, 0.7)                                   # 상위 30% 의 경계값
keep = cv2.GaussianBlur((near >= th).astype(np.float32), (0, 0), 5)[..., None]   # 경계를 부드럽게
portrait = (frame * keep + cv2.GaussianBlur(frame, (0, 0), 10) * (1 - keep)).astype(np.uint8)
show(portrait)

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| 가림막이 엉망 | RGB 로 바꾸지 않았거나 정규화를 빠뜨렸다 |
| `operands could not be broadcast` | 마스크에 채널 축을 안 붙였다 → `[..., None]` |
| 결과 사진이 검거나 하얗게 날아감 | 0~1 을 0~255 로 안 바꿨거나 `clip` 을 빠뜨렸다 |
| 화질 4배 결과에 회색 띠 | 채워 넣은 부분을 잘라 내지 않았다 |

## 정리
- 사진→사진 모델의 후처리: 0~1 로 펴기 → 원래 크기로 → 사진(0~255, uint8)으로.
- 정규화(평균 · 표준편차)는 모델이 배운 입력 범위에 맞추는 일이다.
- 마스크 · 깊이는 그 자체로 끝이 아니라 **합성 재료**다 (배경 바꾸기 · 흐리기 · 인물 모드).

## 해 볼 것
1. 배경을 흰색 대신 다른 사진으로 바꿔 보자 (`frame2 = grab()` 후 크기를 맞춰 합성).
2. 인물 사진 모드의 0.7 을 0.5 · 0.9 로 바꿔 보자.
3. 세 모델을 `"CPU"` 와 `"GPU"` 에서 각각 재 보자. 어느 모델이 GPU 덕을 가장 크게 보나?
4. 깊이로 "카메라에 가까이 온 순간" 을 알아내 보자.
   <details><summary>힌트</summary>

   사진 가운데 영역의 `d`(원래 값, 펴기 전) 평균이 갑자기 커지면 가까이 온 것이다. 펴기 전 값은 사진끼리 비교할 수 있다(대략).
   </details>

---
# 🛠 미니 프로젝트 — 가상 배경 카메라

**만들 것**: 화상회의처럼 웹캠 영상의 **배경을 흐리거나 단색으로 바꾸는** 실시간 카메라. 15초 동안 돌린다.

**쓰는 것**: `u2` · `normalized` (이 노트북의 배경 지우기), 알파 합성, `live`

**속도 요령**: 마스크는 320 x 320 으로 작게 만들어 키우므로, 사진 크기가 커져도 모델 시간은 그대로다. 대신 합성(곱하기)은 사진 크기만큼 든다.

In [ ]:
from IPython.display import Image as IPImage

def live(process, seconds=15, index=0, width=640):
    """웹캠 영상을 seconds 초 동안 process(사진) → 사진 으로 바꿔 노트북 안에 보여 준다. 멈추려면 I I."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    still = None if cap.isOpened() and cap.read()[0] else cv2.imread(PHOTO)   # 웹캠이 없으면 사진으로
    handle = display(None, display_id=True)
    t0, n, out = time.time(), 0, None
    try:
        while time.time() - t0 < seconds and (still is None or n < 10):
            if still is None:
                ok, f = cap.read()
                if not ok:
                    break
            else:
                f = still.copy()
            out = process(f)
            s = width / out.shape[1]
            small = cv2.resize(out, (width, int(out.shape[0] * s))) if s < 1 else out
            handle.update(IPImage(data=cv2.imencode(".jpg", small)[1].tobytes()))
            n += 1
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
    print(f"{n}장 · {n / max(time.time() - t0, 1e-6):.1f} FPS" + (" (웹캠이 없어 사진으로 흉내)" if still is not None else ""))
    return out

In [ ]:
MODE = "blur"                 # "blur" 흐리게 · "color" 단색 · "photo" 다른 사진
BG_COLOR = (90, 160, 60)      # 단색일 때 (B, G, R)
bg_photo = None               # "photo" 일 때 쓸 사진 — 예: cv2.imread("beach.jpg")

def virtual_bg(f):
    h, w = f.shape[:2]
    m = u2(normalized(f, 320))[u2.output(0)].squeeze().astype(np.float32)
    m = (m - m.min()) / (m.max() - m.min() + 1e-8)
    m = cv2.resize(m, (w, h))[..., None]
    if MODE == "blur":
        bg = cv2.GaussianBlur(f, (0, 0), 15)
    elif MODE == "photo" and bg_photo is not None:
        bg = cv2.resize(bg_photo, (w, h))
    else:
        bg = np.full_like(f, BG_COLOR)
    return (f * m + bg * (1 - m)).astype(np.uint8)

live(virtual_bg, seconds=15)

**더 해 보기**
- 마스크를 매 장 새로 만들지 말고 2장에 한 번만 만들어 FPS 올리기 (마스크 재사용)
- 깊이(5절)를 써서 "가까운 것만 선명" 하게 — 마스크 대신 깊이로 합성
- 배경에 시계 글자 넣기 (`cv2.putText`)